## 1. Installation et imports

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 90.5 MB/s eta 0:00:00


In [ ]:
# À exécuter une seule fois (Colab). faiss-cpu suffit : ~300 vecteurs, recherche exacte.
# !pip install -q pymupdf sentence-transformers faiss-cpu

In [ ]:
import os, re, json, math, time, hashlib, unicodedata, platform, random, gc, textwrap
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pymupdf          # PyMuPDF (import "fitz" dans les anciennes versions)
import faiss

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)
print("python", platform.python_version(), "| pymupdf", pymupdf.__doc__.split()[1] if pymupdf.__doc__ else "?", "| faiss", faiss.__version__)

python 3.13.15 | pymupdf 1.28.2: | faiss 1.15.1


## 2. Configuration


In [ ]:
PDF_PATH = Path("cnil_guide_securite_personnelle.pdf")
TEST_SET_PATH = Path("test_set_b1.jsonl")
DEV_SET_PATH = Path("dev_retrieval_b1.jsonl")
OUTPUT_DIR = Path("outputs/step2_index")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# La check-list finale (pages 59-62) répète les mesures des fiches sous forme de tableau :
# l'indexer ajoute des doublons et du bruit, sans information nouvelle. Désactivée par défaut.
INDEX_CHECKLIST = False

TOP_K_MAX = 10            # profondeur de recherche évaluée

# Titres officiels des fiches (page « sommaire » du PDF). Servent d'en-têtes contextuels
# et de contrôle croisé avec les titres détectés dans le texte.
FICHE_TITLES = {
    0: "Présentation et avant-propos",
    1: "Piloter la sécurité des données",
    2: "Définir un cadre pour les utilisateurs",
    3: "Impliquer et former les utilisateurs",
    4: "Authentifier les utilisateurs",
    5: "Gérer les habilitations",
    6: "Sécuriser les postes de travail",
    7: "Sécuriser l'informatique mobile",
    8: "Protéger le réseau informatique",
    9: "Sécuriser les serveurs",
    10: "Sécuriser les sites web",
    11: "Encadrer les développements informatiques",
    12: "Protéger les locaux",
    13: "Sécuriser les échanges avec l'extérieur",
    14: "Gérer la sous-traitance",
    15: "Encadrer la maintenance et la fin de vie des matériels et logiciels",
    16: "Tracer les opérations",
    17: "Sauvegarder",
    18: "Prévoir la continuité et la reprise d'activité",
    19: "Gérer les incidents et les violations",
    20: "Analyse de risques",
    21: "Chiffrement, hachage, signature",
    22: "Cloud : informatique en nuage",
    23: "Applications mobiles : conception et développement",
    24: "Intelligence artificielle : conception et apprentissage",
    25: "API : interfaces de programmation applicative",
}
SECTION_LABELS = {
    "intro": "Introduction",
    "precautions": "Les précautions élémentaires",
    "interdits": "Ce qu'il ne faut pas faire",
    "plus_loin": "Pour aller plus loin",
    "notes": "Notes de bas de page",
    "checklist": "Check-list d'évaluation",
}

## 3. Utilitaires de texte

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", s.lower().replace("’", "'"))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s  ]+", " ", s).strip()

def clean_inline(s):
    """Nettoyage d'une ligne : ligatures (NFKC), caractères de contrôle parasites du PDF, apostrophes, espaces."""
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\x07", "").replace("­", "").replace("​", "")   # \x07 : glyphe de puce mal encodé
    s = s.replace("’", "'").replace("‘", "'")
    return re.sub(r"[ \t]+", " ", s).strip()

# Césure en fin de ligne : on recolle les mots, sauf si le début est un préfixe de mot composé.
KEEP_HYPHEN_PREFIXES = ("sous-", "lui-", "non-", "pare-", "anti-", "peer-", "cross-", "ex-", "mi-", "tout-", "post-", "co-")

def join_lines(prev, nxt):
    if prev.endswith("-") and not prev.endswith(" -") and re.match(r"[a-zàâçéèêëîïôûùüÿœ]", nxt):
        left = prev.split()[-1].lower()
        return prev + nxt if left.startswith(KEEP_HYPHEN_PREFIXES) else prev[:-1] + nxt
    return prev + " " + nxt

# Artefacts d'extraction propres à ce PDF (constatés par inspection du texte brut ; voir contrôle section 5.3).
TEXT_FIXES = {
    "soustraitan": "sous-traitan",   # « sous-traitance » : trait d'union perdu à la césure
    "luimême": "lui-même",
    "(104)": "(10^4)",               # exposants aplatis dans la note 13 (entropie)
    "(213)": "(2^13)",
}
TITLE_FIXES = {"ARTICIFICIELLE": "ARTIFICIELLE"}   # coquille du PDF dans le titre de la fiche 24

def apply_text_fixes(s):
    for a, b in TEXT_FIXES.items():
        s = s.replace(a, b)
    return s

## 4. Extraction du PDF

In [ ]:
FOOTNOTE_START = re.compile(r"^(\d{1,3}(?:\.\d)?)\s+(\S.*)$")

def page_rows(page):
    rows = []
    for b in page.get_text("dict")["blocks"]:
        for l in b.get("lines", []):
            body_spans = [s for s in l["spans"] if not (s["flags"] & 1)]       # flags & 1 : exposant
            if not body_spans:
                continue
            size = round(max(s["size"] for s in body_spans), 1)
            # dans les notes (police <= 6 pt) on garde les exposants : ce sont de vrais exposants (10^4 ...)
            if size <= 6.0:
                text = "".join(s["text"] for s in l["spans"])
            else:
                # appel de note supprimé ; on garde l'espace éventuel qui l'entourait (« CNIL17.1 dans » -> « CNIL dans »)
                text = "".join((s["text"] if not (s["flags"] & 1) else (" " if s["text"] != s["text"].rstrip() else ""))
                               for s in l["spans"])
            text = clean_inline(text)
            if text:
                rows.append({"y": l["bbox"][1], "x": l["bbox"][0], "size": size, "text": text})
    rows.sort(key=lambda r: (round(r["y"] / 4), r["x"]))   # ordre de lecture ; tolérance de 4 pt pour les cellules de tableau
    return rows

def classify_rows(rows):
    """Sépare les lignes d'une page : titres, numéro imprimé, tag, notes, tableau, corps."""
    out = {"heads": [], "printed": None, "tag": False, "notes": [], "table": [], "body": [], "sizes": set()}
    fn_idx = next((i for i, r in enumerate(rows) if r["size"] <= 6.0 and FOOTNOTE_START.match(r["text"])), None)
    for i, r in enumerate(rows):
        out["sizes"].add(r["size"])
        if r["size"] == 10.0 and re.fullmatch(r"\d{1,3}", r["text"]):
            out["printed"] = int(r["text"])
        elif r["size"] >= 13.0:
            out["heads"].append(r)
        elif r["size"] == 6.6 and r["text"].upper() == "POUR ALLER PLUS LOIN":
            out["tag"] = True
            out["tag_y"] = r["y"]
        elif fn_idx is not None and i >= fn_idx and r["size"] <= 6.0:
            out["notes"].append(r)
        elif r["size"] == 7.0:
            out["table"].append(r)
        else:
            out["body"].append(r)
    return out

doc = pymupdf.open(PDF_PATH)
PDF_SHA256 = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
raw_pages = []
for i, page in enumerate(doc):
    c = classify_rows(page_rows(page))
    c["pdf_page"] = i + 1
    raw_pages.append(c)
print(len(raw_pages), "pages | SHA-256 du PDF :", PDF_SHA256[:16], "...")

64 pages | SHA-256 du PDF : 5602bd27bc00e5c5 ...


### 4.1 Numérotation des pages (citations)

In [ ]:
pairs = [(c["pdf_page"], c["printed"]) for c in raw_pages if c["printed"] is not None]
offset = Counter(p - q for p, q in pairs).most_common(1)[0][0]
anomalies = [(p, q) for p, q in pairs if p - q != offset]
print(f"Décalage dominant : page imprimée = page PDF − {offset}")
print("Anomalies de numérotation (page PDF, numéro imprimé) :", anomalies)
for c in raw_pages:
    c["page"] = c["pdf_page"] - offset          # numéro imprimé « théorique », utilisé partout ensuite

Décalage dominant : page imprimée = page PDF − 2
Anomalies de numérotation (page PDF, numéro imprimé) : [(63, 6)]


### 4.2 Type de chaque page et suivi de la fiche courante

In [ ]:
def heading_text(c):
    t = " ".join(r["text"] for r in c["heads"] if r["size"] < 30)
    for a, b in TITLE_FIXES.items():
        t = t.replace(a, b)
    return t

def page_kind(c, idx, n_pages, checklist_started):
    if idx == 0:
        return "cover"
    if idx == n_pages - 1 and not c["body"]:
        return "back_cover"
    if 7.8 in c["sizes"]:
        return "toc"
    if any(r["size"] >= 30 for r in c["heads"]) and not c["body"]:
        return "divider"
    if "ÉVALUER LE NIVEAU DE SÉCURITÉ" in heading_text(c).upper() or checklist_started:
        return "checklist"
    return "content"

checklist_started = False
for i, c in enumerate(raw_pages):
    c["kind"] = page_kind(c, i, len(raw_pages), checklist_started)
    checklist_started = checklist_started or c["kind"] == "checklist"
    # le texte d'une dernière page non classée (ex. coordonnées) ne doit pas polluer l'index
    if c["kind"] == "back_cover" or (i == len(raw_pages) - 1):
        c["kind"] = "back_cover"

# Page 2 du PDF : présentation du guide (sans titre ni numéro) ; on la rattache à la « fiche 0 »
print(pd.Series([c["kind"] for c in raw_pages]).value_counts().to_string())
fiche_heads = [(c["pdf_page"], re.match(r"FICHE\s*(\d+)", heading_text(c).upper())) for c in raw_pages if c["kind"] == "content"]
fiche_heads = [(p, int(m.group(1))) for p, m in fiche_heads if m]
assert [f for _, f in fiche_heads] == list(range(1, 26)), f"fiches détectées : {[f for _, f in fiche_heads]}"
print("25 fiches détectées, dans l'ordre. Pages PDF de début :", [p for p, _ in fiche_heads])

# contrôle croisé titres détectés / titres officiels
def fiche_title_text(c):
    """Titre de fiche seul (lignes de taille 14, éventuellement sur deux lignes)."""
    t = " ".join(r["text"] for r in c["heads"] if r["size"] == 14.0)
    for a, b in TITLE_FIXES.items():
        t = t.replace(a, b)
    return t

def _title_from_heading(h):
    return norm(re.sub(r"^FICHE\s*\d+\s*[-–—]\s*", "", h, flags=re.I))
bad_titles = []
for c in raw_pages:
    m = re.match(r"FICHE\s*(\d+)", fiche_title_text(c).upper()) if c["kind"] == "content" else None
    if m and _title_from_heading(fiche_title_text(c)).replace(" ", "") != norm(FICHE_TITLES[int(m.group(1))]).replace(" ", ""):
        bad_titles.append((int(m.group(1)), fiche_title_text(c)))
print("Titres divergents (attendu : aucun) :", bad_titles)

content       53
divider        5
checklist      3
cover          1
toc            1
back_cover     1
25 fiches détectées, dans l'ordre. Pages PDF de début : [5, 9, 11, 13, 16, 18, 20, 22, 24, 26, 28, 30, 33, 35, 37, 40, 42, 43, 44, 47, 50, 52, 55, 57, 59]
Titres divergents (attendu : aucun) : []


## 5. Segmentation en « éléments » (puces) avec métadonnées

In [ ]:
BULLET = re.compile(r"^[•·]\s*(.*)$")
NUMBERED = re.compile(r"^(\d{1,2}\.)\s+(\S.*)$")
SUBBULLET = re.compile(r"^-\s+(.*)$")
SECTION_BY_TITLE = {"les precautions elementaires": "precautions", "ce qu'il ne faut pas faire": "interdits"}

units, notes_units = [], []     # éléments du corps / éléments de notes
state = {"fiche": 0, "section": "intro"}
cur = None                      # élément en cours de construction
last_y = None

def close_unit():
    global cur
    if cur is not None and cur["text"].strip():
        cur["text"] = apply_text_fixes(re.sub(r"[ ]+", " ", cur["text"]).strip())
        units.append(cur)
    cur = None

def new_unit(text, page, y, kind):
    global cur
    close_unit()
    cur = {"fiche": state["fiche"], "section": state["section"], "kind": kind, "text": text, "pages": [page], "y": y}

page_texts = {}      # texte propre de chaque page (corps + notes) : sert au contrôle du jeu de test

for c in raw_pages:
    if c["kind"] in ("cover", "back_cover", "toc", "divider"):
        continue
    page = c["page"]
    if c["kind"] == "checklist":
        if INDEX_CHECKLIST:
            allrows = sorted(c["body"] + c["table"], key=lambda r: (round(r["y"] / 4), r["x"]))
            state_cl = {"fiche": -1, "section": "checklist"}
            close_unit()
            cur = {"fiche": -1, "section": "checklist", "kind": "table",
                   "text": " | ".join(r["text"] for r in allrows), "pages": [page], "y": 0}
            close_unit()
        continue

    # --- titres de la page : mise à jour de l'état (fiche, section)
    head_events = []
    for r in c["heads"]:
        if r["size"] >= 30:
            continue
        t = r["text"].upper()
        m = re.match(r"FICHE\s*(\d+)", t)
        if m:
            head_events.append((r["y"], "fiche", int(m.group(1))))
        elif t.startswith("AVANT-PROPOS"):
            head_events.append((r["y"], "fiche", 0))
            head_events.append((r["y"] + 0.1, "section", "intro"))
        elif norm(r["text"]) in SECTION_BY_TITLE:
            head_events.append((r["y"], "section", SECTION_BY_TITLE[norm(r["text"])]))
    if c["tag"]:
        head_events.append((c["tag_y"], "section", "plus_loin"))
    head_events.sort()

    # --- corps + tableau, dans l'ordre vertical, avec les événements de titre intercalés
    body = sorted(c["body"] + c["table"], key=lambda r: (round(r["y"] / 4), r["x"]))
    stream = [(r["y"], "row", r) for r in body] + [(y, "head", (k, v)) for y, k, v in head_events]
    stream.sort(key=lambda e: (round(e[0] / 4), 0 if e[1] == "head" else 1))
    first_row_on_page = True
    last_y = None
    in_table = False
    for y, typ, payload in stream:
        if typ == "head":
            kind_h, val = payload
            close_unit()
            if kind_h == "fiche":
                state["fiche"], state["section"] = val, "intro"
            else:
                state["section"] = val
            continue
        r = payload
        text = r["text"]
        if r["size"] == 7.0:                       # cellule de tableau (fiche 20)
            if not in_table:
                new_unit("Tableau de formalisation de l'analyse de risques : " + text, page, y, "table"); in_table = True
            else:
                cur["text"] += " | " + text
            continue
        in_table = False
        m_b, m_n, m_s = BULLET.match(text), NUMBERED.match(text), SUBBULLET.match(text)
        if text in ("]", ")") and cur is not None:    # ligne réduite à un crochet fermant (clause encadrée)
            cur["text"] += text
        elif m_b:
            new_unit(m_b.group(1), page, y, "bullet")
        elif m_n and (cur is None or cur["kind"] != "bullet" or state["fiche"] == 2):
            new_unit(m_n.group(1) + " " + m_n.group(2), page, y, "bullet")
        elif m_s:
            if cur is None:
                new_unit(m_s.group(1), page, y, "bullet")
            else:
                cur["text"] += "\n- " + m_s.group(1)
        else:
            if cur is None:
                new_unit(text, page, y, "para")
            elif first_row_on_page:
                ended = re.search(r"[.!?»]$", cur["text"]) is not None
                if ended and text[:1].isupper():
                    new_unit(text, page, y, "para")
                else:
                    cur["text"] = join_lines(cur["text"], text)
                    if page not in cur["pages"]: cur["pages"].append(page)
            elif last_y is not None and (y - last_y) > 14:
                new_unit(text, page, y, "para")
            else:
                cur["text"] = join_lines(cur["text"], text)
        first_row_on_page = False
        last_y = y
        if cur is not None and page not in cur["pages"]:
            cur["pages"].append(page)
    # un élément ne doit pas « déborder » sur une page d'une autre fiche : la fermeture se fait aux titres

    # --- notes de bas de page de cette page
    notes = []
    for r in c["notes"]:
        m = FOOTNOTE_START.match(r["text"])
        if m:
            notes.append([m.group(1), m.group(2)])
        elif notes:
            notes[-1][1] = join_lines(notes[-1][1], r["text"])
    for num, txt in notes:
        notes_units.append({"fiche": state["fiche"], "section": "notes", "kind": "note",
                            "text": apply_text_fixes(f"[{num}] {txt}"), "pages": [page], "y": 1e6})
    page_texts[page] = apply_text_fixes(
        " ".join(r["text"] for r in body) + " " + " ".join(f"[{n}] {t}" for n, t in notes))
close_unit()

print(f"{len(units)} éléments de corps | {len(notes_units)} notes | pages indexables : {len(page_texts)}")
unit_df = pd.DataFrame(units)
display(unit_df.groupby(["fiche", "section"]).agg(n=("text", "size"), chars=("text", lambda s: int(s.str.len().sum()))).unstack(fill_value=0))

455 éléments de corps | 134 notes | pages indexables : 53


n                                 chars                            
section interdits intro plus_loin precautions interdits intro plus_loin precautions
fiche                                                                              
0               0     9         0           0         0  2981         0           0
1               6     2         5           6       738   341      2281        3367
2               3     2         4           7       463   279      2920        2240
3               4     2         3           5       661   348       850        2162
4               5     2        11           4       824   659      3173        2790
5               5     2         1           4       560   254       644         890
6               2     2        11           9       218   285      1868        1938
7               3     2         7           7       875   437      1619        1911
8               3     2         6           7       249   385      1160        1463
9               5     2         8          10       553   223      1525        1520
10              5     2         3           8       504   196       851        2093
11              2     2         9           9       293   456      2657        1796
12              2     2         2           9       655   384       901        1194
13              2     2         5           4       255   665      1195        1370
14              2     2         7           3       389   399      1281        1925
15              3     2        11           6       365   291      1610         785
16              3     2         4           8       533   513       661        2176
17              4     2         4           6       680   232       593         903
18              4     2         4           4       306   499       676         667
19              3     2         5           7       492   512       828        1700
20              3     2         6           9       345   541      1398        4537
21              4     2         6           5       761   377      1147        1242
22             11     2         6           7      1499   669      1260        2687
23              2     2         5           6       604   391      2306        1583
24              3     2         6           7       519   740      1406        2381
25              2     2         4           5       257   523       738        1054

### 5.1 Contrôles de cohérence de l'extraction

In [ ]:
fiches_found = sorted({u["fiche"] for u in units if u["fiche"] > 0})
assert fiches_found == list(range(1, 26)), fiches_found
for f in range(1, 26):
    secs = {u["section"] for u in units if u["fiche"] == f}
    assert "precautions" in secs, f"fiche {f} sans « précautions élémentaires »"
print("OK : 25 fiches, toutes avec section « précautions élémentaires ».")
print("Fiches sans « Ce qu'il ne faut pas faire » :", [f for f in range(1, 26) if not any(u['fiche'] == f and u['section'] == 'interdits' for u in units)])
print("Éléments très courts (< 25 car.) :", [(u['fiche'], u['pages'], u['text']) for u in units if len(u['text']) < 25][:8])
print("Éléments très longs (> 1200 car.) :", [(u['fiche'], u['pages'], len(u['text'])) for u in units if len(u['text']) > 1200])

OK : 25 fiches, toutes avec section « précautions élémentaires ».
Fiches sans « Ce qu'il ne faut pas faire » : []
Éléments très courts (< 25 car.) : [(18, [41], "Se considérer à l'abri.")]
Éléments très longs (> 1200 car.) : [(2, [7], 1251), (2, [8], 2430), (4, [11], 1471), (20, [45, 46], 2297)]


### 5.2 Échantillon à relire à l'œil



In [ ]:
def show_unit(u, width=330):
    print(f"[fiche {u['fiche']:>2} | {u['section']:<11} | p.{','.join(map(str, u['pages']))} | {u['kind']}] "
          + textwrap.shorten(u["text"].replace("\n", " ⏎ "), width=width))

for f, sec in [(4, "precautions"), (4, "plus_loin"), (11, "precautions"), (24, "interdits")]:
    for u in [u for u in units if u["fiche"] == f and u["section"] == sec][:3]:
        show_unit(u)
print("\n--- notes (échantillon) ---")
for u in [u for u in notes_units if u["fiche"] in (4, 24, 17)][:4]:
    show_unit(u)

[fiche  4 | precautions | p.11 | bullet] Définir un identifiant unique par utilisateur et interdire les comptes partagés entre plusieurs utilisateurs. Dans le cas où l'utilisation d'identifiants génériques ou partagés est incontournable, exiger une validation de la hiérarchie, mettre en œuvre des moyens pour tracer les actions associées à ces identifiants et [...]
[fiche  4 | precautions | p.11 | bullet] Respecter la recommandation de la CNIL dans le cas d'une authentification des utilisateurs basée sur des mots de passe, notamment en appliquant les règles suivantes : ⏎ - stocker uniquement l'empreinte des mots de passe, obtenue selon des techniques à l'état de l'art ; ⏎ - ne pas demander le renouvellement périodique des [...]
[fiche  4 | precautions | p.12 | bullet] Accompagner les utilisateurs pour le choix d'un mot de passe robuste : ⏎ - en sensibilisant sur les moyens mnémotechniques ; ⏎ - en incitant à l'utilisation de gestionnaires de mots de passe et en formant à leur utilisatio

### 5.3 Recherche d'artefacts d'extraction restants


In [ ]:
all_text = "\n".join(u["text"] for u in units + notes_units)
tokens = Counter(re.findall(r"[a-zàâçéèêëîïôûùüÿœ]{3,}", all_text.lower()))
suspects = sorted(
    [t for t, n in tokens.items() if n == 1 and (t.endswith("nelles") or t.endswith("ment") and len(t) > 16 or t.startswith("sous") and "-" not in t)],
)
print("Mots suspects (à vérifier) :", suspects[:40])
print("Césures restantes (« mot- mot ») :", re.findall(r"\w+- \w+", all_text)[:10])

Mots suspects (à vérifier) : ['approvisionnement', 'cryptographiquement', 'fonctionnelles', 'nelles', 'soussigné']
Césures restantes (« mot- mot ») : []


## 6. Chunking structurel



In [ ]:
def section_label(sec):
    return SECTION_LABELS.get(sec, sec)

def make_header(fiche, section):
    if fiche == 0:
        return "Présentation du guide"
    return f"Fiche {fiche} — {FICHE_TITLES[fiche]} — {section_label(section)}"

def hard_split(text, max_chars):
    out = []
    while len(text) > max_chars:
        cut = text.rfind(" ", 0, max_chars)
        cut = cut if cut > max_chars // 2 else max_chars
        out.append(text[:cut].strip()); text = text[cut:].strip()
    return out + [text] if text else out

def split_unit_text(text, max_chars):
    """Découpe un élément trop long. Retourne une liste de morceaux (chacun <= max_chars sauf cas pathologique)."""
    if len(text) <= max_chars:
        return [text]
    lines = text.split("\n")
    if len(lines) > 1:                                   # coupe aux sous-puces, en répétant l'amorce
        lead, pieces, mode = lines[0], lines[1:], "lines"
    else:                                                # coupe aux phrases
        lead, mode = "", "sent"
        pieces = re.split(r"(?<=[.!?;:])\s+(?=[A-ZÉÈÀÂÎÔÛ«(\[])", text)
    out, buf = [], lead
    for p in pieces:
        sep = "\n" if mode == "lines" else " "
        cand = (buf + sep + p) if buf else p
        if len(cand) > max_chars and buf and buf != lead:
            out.append(buf)
            buf = (lead + sep + p) if mode == "lines" else p
        else:
            buf = cand
    if buf and buf != lead:
        out.append(buf)
    final = []
    for t in out:
        final += hard_split(t, max_chars) if len(t) > max_chars else [t]
    return final

def pack_group(pieces, max_chars, overlap_units, min_chars):
    """pieces : liste de (texte, pages, est_un_morceau_coupé). Retourne une liste de listes de pièces (un chunk = une liste)."""
    chunks, cur, cur_new, cur_len = [], [], [], 0
    for piece in pieces:
        t = piece[0]
        add = len(t) + (2 if cur else 0)
        if cur and cur_len + add > max_chars:
            chunks.append(cur)
            carry = [p for p in cur[-overlap_units:] if not p[2]] if overlap_units else []
            carry_len = sum(len(p[0]) for p in carry) + 2 * max(len(carry) - 1, 0)
            if carry and carry_len + len(t) + 2 <= max_chars:
                cur, cur_len = list(carry), carry_len
            else:
                cur, cur_len = [], 0
            cur_new = []
            add = len(t) + (2 if cur else 0)
        cur.append(piece); cur_new.append(piece); cur_len += add
    if cur:
        new_len = sum(len(p[0]) for p in cur_new)
        if chunks and new_len < min_chars and sum(len(p[0]) for p in chunks[-1]) + new_len <= 1.3 * max_chars:
            chunks[-1] = chunks[-1] + cur_new        # le petit reste rejoint le chunk précédent
        else:
            chunks.append(cur)
    return chunks

def make_chunks(units, notes_units, max_chars=900, overlap_units=1, min_chars=120):
    groups = []
    for u in units:
        sec_group = "precautions" if u["section"] in ("intro", "precautions") else u["section"]
        key = (u["fiche"], sec_group)
        if groups and groups[-1][0] == key:
            groups[-1][1].append(u)
        else:
            groups.append((key, [u]))
    for u in notes_units:                                 # notes : un groupe par (fiche, page)
        key = (u["fiche"], "notes", u["pages"][0])
        if groups and groups[-1][0] == key:
            groups[-1][1].append(u)
        else:
            groups.append((key, [u]))
    chunks = []
    for key, us in groups:
        is_note = key[1] == "notes"
        pieces = []
        for u in us:
            parts = split_unit_text(u["text"], max_chars)
            pieces += [(t, u["pages"], len(parts) > 1, u["section"]) for t in parts]
        for ch in pack_group([p[:3] for p in pieces], max_chars, 0 if is_note else overlap_units, min_chars):
            text = "\n\n".join(p[0] for p in ch) if not is_note else "\n".join(p[0] for p in ch)
            pages = sorted({pg for p in ch for pg in p[1]})
            sections = {sp[3] for sp in pieces if any(sp[0] == p[0] for p in ch)}
            sec = "notes" if is_note else ("precautions" if "precautions" in sections else (sorted(sections)[0] if sections else key[1]))
            fiche = key[0]
            hdr = make_header(fiche, sec)
            chunks.append({"fiche": fiche, "fiche_title": FICHE_TITLES[fiche], "section": sec, "section_label": section_label(sec),
                           "pages": pages, "text": text, "embed_text": hdr + "\n" + text, "n_chars": len(text),
                           "kind": "notes" if is_note else "body"})
    for i, c in enumerate(chunks):
        c["chunk_id"] = f"c{i:04d}"
    return chunks

def cite(c):
    p = c["pages"]
    pages = f"p. {p[0]}" if len(p) == 1 else f"p. {p[0]}-{p[-1]}"
    where = "Présentation du guide" if c["fiche"] == 0 else f"Fiche {c['fiche']} « {c['fiche_title']} »"
    return f"{where}, {c['section_label']}, {pages}"

def coverage_check(chunks, units, notes_units):
    """Vérifie qu'aucun contenu n'est perdu : le début de chaque élément doit se retrouver dans au moins un chunk."""
    blob = norm(" ".join(c["text"] for c in chunks))
    lost = [u for u in units + notes_units if norm(u["text"].split("\n")[0])[:50] not in blob]
    return lost

DEFAULT_CHUNK_CFG = dict(max_chars=900, overlap_units=1)
demo_chunks = make_chunks(units, notes_units, **DEFAULT_CHUNK_CFG)
lost = coverage_check(demo_chunks, units, notes_units)
print(f"{len(demo_chunks)} chunks (config par défaut {DEFAULT_CHUNK_CFG}) | éléments perdus : {len(lost)}")
assert not lost, lost[:3]
s = pd.Series([c["n_chars"] for c in demo_chunks])
print("longueur des chunks (car.) :", s.describe().round(0).to_dict())
for c in [demo_chunks[i] for i in (3, 40, 120, len(demo_chunks) - 5)]:
    print(f"\n[{c['chunk_id']}] {cite(c)} ({c['n_chars']} car.)\n{textwrap.shorten(c['text'].replace(chr(10), ' ⏎ '), 380)}")

239 chunks (config par défaut {'max_chars': 900, 'overlap_units': 1}) | éléments perdus : 0
longueur des chunks (car.) : {'count': 239.0, 'mean': 625.0, 'std': 225.0, 'min': 34.0, '25%': 474.0, '50%': 682.0, '75%': 818.0, 'max': 898.0}

[c0003] Présentation du guide, Introduction, p. 2 (740 car.)
Il est cependant difficile, pour les non-spécialistes de la sécurité informatique, de mettre en œuvre une telle démarche et de s'assurer que le niveau de sécurité des traitements dont ils sont responsables est suffisant. ⏎ ⏎ Pour aider dans la mise en conformité, ce guide présente un ensemble de préconisations regroupées par fiches thématiques. Chaque fiche est structurée [...]

[c0040] Fiche 4 « Authentifier les utilisateurs », Les précautions élémentaires, p. 11 (403 car.)
Définir un identifiant unique par utilisateur et interdire les comptes partagés entre plusieurs utilisateurs. Dans le cas où l'utilisation d'identifiants génériques ou partagés est incontournable, exiger une validation de 

## 7. Jeux de données et contrôles


In [ ]:
def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

dev = pd.DataFrame(load_jsonl(DEV_SET_PATH))
test = pd.DataFrame(load_jsonl(TEST_SET_PATH))
DEV_SHA256 = hashlib.sha256(DEV_SET_PATH.read_bytes()).hexdigest()
TEST_SET_SHA256 = hashlib.sha256(TEST_SET_PATH.read_bytes()).hexdigest()
assert not ({norm(q) for q in dev.question} & {norm(q) for q in test.question}), "des questions sont communes au dev et au test"
dev_in, dev_ooc = dev[dev.category == "in_corpus"].reset_index(drop=True), dev[dev.category != "in_corpus"].reset_index(drop=True)
test_in, test_ooc = test[test.category == "in_corpus"].reset_index(drop=True), test[test.category != "in_corpus"].reset_index(drop=True)
print(f"dev : {len(dev_in)} en corpus + {len(dev_ooc)} hors corpus | test : {len(test_in)} + {len(test_ooc)} | SHA-256 test : {TEST_SET_SHA256[:16]}...")

# Contrôle des références du test contre le texte extrait
fiche_norm = defaultdict(str)
for u in units + notes_units:
    fiche_norm[u["fiche"]] += " " + norm(u["text"])
page_norm = {p: norm(t) for p, t in page_texts.items()}
def fact_in(alts, text):
    return any(re.search(norm(a), text) for a in alts)
problems = []
for _, r in test_in.iterrows():
    ft = " ".join(fiche_norm[f] for f in r.gold_fiches); pt = " ".join(page_norm.get(p, "") for p in r.gold_pages)
    miss_f = [a for a in r.key_facts if not fact_in(a, ft)]; miss_p = [a for a in r.key_facts if not fact_in(a, pt)]
    if miss_f or miss_p:
        problems.append((r.id, "fiches" if miss_f else "", "pages" if miss_p else ""))
print("Questions du test dont un fait-clé est introuvable dans les fiches/pages de référence :", problems or "aucune")

dev : 62 en corpus + 13 hors corpus | test : 33 + 17 | SHA-256 test : f9ce59c626039960...
Questions du test dont un fait-clé est introuvable dans les fiches/pages de référence : aucune


## 8. Outils de recherche : BM25, embeddings, fusion



In [ ]:
STOPWORDS = set("le la les l un une des de du d et ou a au aux en dans par pour sur sous avec sans que qui quoi est sont etre ce cet cette ces se sa son ses "
                "leur leurs ne pas plus il ils elle elles on y c s n qu j m t si comme mais donc car faut faire peut doit quel quelle quels quelles".split())

def tokenize(text):
    toks = re.findall(r"[a-z0-9]+", norm(text))
    return [t[:-1] if len(t) > 3 and t.endswith("s") else t for t in toks if t not in STOPWORDS]   # pluriels naïfs

class BM25:
    """BM25 Okapi minimal (k1=1.5, b=0.75). ~300 documents : une implémentation simple suffit."""
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [Counter(d) for d in docs_tokens]
        self.len = np.array([len(d) for d in docs_tokens], dtype=float)
        self.avg = self.len.mean()
        df = Counter(t for d in self.docs for t in d)
        n = len(docs_tokens)
        self.idf = {t: math.log(1 + (n - f + 0.5) / (f + 0.5)) for t, f in df.items()}
    def scores(self, q_tokens):
        s = np.zeros(len(self.docs))
        for t in set(q_tokens):
            if t not in self.idf:
                continue
            tf = np.array([d.get(t, 0) for d in self.docs], dtype=float)
            s += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.len / self.avg))
        return s

def rrf_fuse(rank_lists, k=60, topn=TOP_K_MAX):
    """rank_lists : liste de listes de chunks classés (une par méthode). Retourne les topn indices fusionnés."""
    sc = defaultdict(float)
    for ranks in rank_lists:
        for pos, j in enumerate(ranks):
            sc[int(j)] += 1.0 / (k + pos + 1)
    return [j for j, _ in sorted(sc.items(), key=lambda x: -x[1])[:topn]]

EMBEDDERS = {
    "e5-base":  {"hf_id": "intfloat/multilingual-e5-base",  "q": "query: ", "p": "passage: "},
    "e5-large": {"hf_id": "intfloat/multilingual-e5-large", "q": "query: ", "p": "passage: "},
    "bge-m3":   {"hf_id": "BAAI/bge-m3",                    "q": "",        "p": ""},
}

class STEmbedder:
    """Enveloppe sentence-transformers. Vecteurs normalisés (produit scalaire = cosinus)."""
    def __init__(self, name):
        import torch
        from sentence_transformers import SentenceTransformer
        self.name, self.cfg = name, EMBEDDERS[name]
        self.device = "cuda" if torch.cuda.is_available() else "cpu"      # CPU possible : lent mais ~300 passages seulement
        self.model = SentenceTransformer(self.cfg["hf_id"], device=self.device)
        self.model.max_seq_length = 512                                   # nos chunks font < 512 tokens
        if self.device == "cuda":
            self.model.half()
    def _enc(self, texts, prefix):
        return self.model.encode([prefix + t for t in texts], batch_size=32, normalize_embeddings=True,
                                 convert_to_numpy=True, show_progress_bar=False).astype("float32")
    def encode_passages(self, texts): return self._enc(texts, self.cfg["p"])
    def encode_queries(self, texts):  return self._enc(texts, self.cfg["q"])
    def close(self):
        import torch
        del self.model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def make_embedder(name):
    return STEmbedder(name)

## 9. Expériences sur le jeu de dev


In [ ]:
CHUNK_GRID = [dict(max_chars=m, overlap_units=o) for m in (500, 900, 1400) for o in (0, 1)]
EMBED_NAMES = ["e5-base", "e5-large", "bge-m3"]
KS = (1, 3, 5, 8, 10)
K_MAX = 8                                  # k maximal admis pour la sélection (règle section 0)
KS_SEL = tuple(k for k in KS if k <= K_MAX)

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),) * 2
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

def fact_matrices(key_facts_list, chunks):
    """Pour chaque question : matrice booléenne (n_faits, n_chunks) = le fait (regex à alternatives) apparaît dans le chunk."""
    texts = [norm(c["text"]) for c in chunks]
    out = []
    for kf in key_facts_list:
        pats = [re.compile("|".join("(?:" + norm(a) + ")" for a in alts)) for alts in kf]
        out.append(np.array([[bool(p.search(t)) for t in texts] for p in pats]))
    return out

def retrieval_metrics(ranked, chunks, fact_mats, gold_fiches):
    """ranked : (nq, K) indices classés ; fact_mats : liste (par question) de matrices (n_faits, n_chunks)."""
    evidence_matrix = np.array([fm.any(axis=0) for fm in fact_mats])           # au moins un fait dans le chunk (diagnostic)
    hits = np.take_along_axis(evidence_matrix, ranked, axis=1)
    res = {f"hit@{k}": float(hits[:, :k].any(axis=1).mean()) for k in KS}
    for k in KS:
        found = [fm[:, ranked[i, :k]].any(axis=1) for i, fm in enumerate(fact_mats)]   # faits retrouvés dans l'UNION des k chunks
        res[f"cov@{k}"] = float(np.mean([f.mean() for f in found]))
        res[f"rep@{k}"] = float(np.mean([f.all() for f in found]))
        res[f"ctx@{k}"] = float(np.mean([sum(len(chunks[j]["text"]) for j in row[:k]) for row in ranked]))
    first = np.where(hits.any(axis=1), hits.argmax(axis=1) + 1, np.inf)
    res["MRR"] = float(np.mean(1.0 / first))
    cf = np.array([c["fiche"] for c in chunks])
    fh = np.array([[cf[j] in g for j in row] for row, g in zip(ranked, gold_fiches)])
    res["fiche_hit@5"] = float(fh[:, :5].any(axis=1).mean())
    res["fiche_prec@5"] = float(fh[:, :5].mean())
    return res

def topn(scores, n):
    return np.argsort(-scores, axis=1)[:, :n]

rows_exp, store, unreach = [], {}, {}      # store : (cfg_id, embedder) -> (P, Q_in, Q_ooc) pour réutilisation en section 10
queries_in, queries_ooc = dev_in.question.tolist(), dev_ooc.question.tolist()

# 1) BM25 + préparation des chunks pour chaque chunking
prepared = {}
for cfg in CHUNK_GRID:
    cid = f"{cfg['max_chars']}/{cfg['overlap_units']}"
    chunks = make_chunks(units, notes_units, **cfg)
    assert not coverage_check(chunks, units, notes_units)
    fm = fact_matrices(dev_in.key_facts.tolist(), chunks)
    ev = np.array([m.any(axis=0) for m in fm])
    unreachable = int(sum((~m.any(axis=1)).any() for m in fm))   # au moins un fait introuvable dans TOUT chunk (coupé par le chunking)
    bm = BM25([tokenize(c["embed_text"]) for c in chunks])
    bm_rank = np.array([np.argsort(-bm.scores(tokenize(q)))[:50] for q in queries_in])
    prepared[cid] = dict(cfg=cfg, chunks=chunks, ev=ev, fm=fm, bm=bm, bm_rank=bm_rank)
    m = retrieval_metrics(bm_rank[:, :TOP_K_MAX], chunks, fm, dev_in.gold_fiches.tolist())
    unreach[cid] = unreachable
    rows_exp.append({"chunking": cid, "embedder": "-", "method": "bm25", "n_chunks": len(chunks), **m})
    print(f"chunking {cid:>6} : {len(chunks)} chunks | preuve coupée par le chunking : {unreachable} question(s)")

# 2) embeddings denses + hybride
for name in EMBED_NAMES:
    try:
        emb = make_embedder(name)
    except Exception as e:
        print(f"[!] embedder {name} indisponible ({type(e).__name__}: {e}) -> ignoré")
        continue
    t0 = time.time()
    Q_in, Q_ooc = emb.encode_queries(queries_in), emb.encode_queries(queries_ooc)
    for cid, pr in prepared.items():
        P = emb.encode_passages([c["embed_text"] for c in pr["chunks"]])
        sc = Q_in @ P.T
        dense_rank = topn(sc, 50)
        store[(cid, name)] = (P, Q_in, Q_ooc)
        gold = dev_in.gold_fiches.tolist()
        m = retrieval_metrics(dense_rank[:, :TOP_K_MAX], pr["chunks"], pr["fm"], gold)
        rows_exp.append({"chunking": cid, "embedder": name, "method": "dense", "n_chunks": len(pr["chunks"]), **m})
        hyb = np.array([rrf_fuse([d, b]) for d, b in zip(dense_rank, pr["bm_rank"])])
        m = retrieval_metrics(hyb, pr["chunks"], pr["fm"], gold)
        rows_exp.append({"chunking": cid, "embedder": name, "method": "hybrid", "n_chunks": len(pr["chunks"]), **m})
    print(f"{name:9s} terminé en {time.time()-t0:.0f}s")
    emb.close()

exp = pd.DataFrame(rows_exp)
exp["preuve_coupée"] = exp.chunking.map(unreach)       # nb de questions dont au moins un fait n'est dans aucun chunk de ce chunking
exp.to_csv(OUTPUT_DIR / "experiments_dev.csv", index=False)
cols = ["chunking", "embedder", "method", "n_chunks", "rep@1", "rep@3", "rep@5", "rep@8", "cov@3", "cov@5", "hit@5", "MRR", "fiche_prec@5", "ctx@5", "preuve_coupée"]
display(exp.sort_values(["rep@5", "cov@5", "MRR"], ascending=False)[cols].round(3).head(15))

chunking  500/0 : 371 chunks | preuve coupée par le chunking : 0 question(s)
chunking  500/1 : 396 chunks | preuve coupée par le chunking : 0 question(s)
chunking  900/0 : 217 chunks | preuve coupée par le chunking : 0 question(s)
chunking  900/1 : 239 chunks | preuve coupée par le chunking : 0 question(s)
chunking 1400/0 : 165 chunks | preuve coupée par le chunking : 0 question(s)
chunking 1400/1 : 171 chunks | preuve coupée par le chunking : 0 question(s)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (34 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

bge-m3    terminé en 11s


,chunking,embedder,method,n_chunks,rep@1,rep@3,rep@5,rep@8,cov@3,cov@5,hit@5,MRR,fiche_prec@5,ctx@5,preuve_coupée
16,1400/1,e5-base,dense,171,0.629,0.919,0.984,1.000,0.931,0.996,1.000,0.795,0.623,4517.065,0
41,1400/1,bge-m3,hybrid,171,0.742,0.968,0.984,0.984,0.968,0.984,0.984,0.869,0.652,4705.790,0
36,900/1,bge-m3,dense,239,0.677,0.887,0.984,1.000,0.909,0.984,0.984,0.852,0.694,3419.097,0
17,1400/1,e5-base,hybrid,171,0.710,0.903,0.984,0.984,0.915,0.984,0.984,0.841,0.652,4730.306,0
29,1400/1,e5-large,hybrid,171,0.694,0.919,0.984,0.984,0.931,0.984,0.984,0.824,0.655,4505.452,0
14,1400/0,e5-base,dense,165,0.629,0.871,0.984,0.984,0.892,0.984,0.984,0.787,0.623,4413.048,0
34,900/0,bge-m3,dense,217,0.645,0.903,0.968,0.984,0.936,0.980,0.984,0.838,0.652,3263.758,0
24,900/1,e5-large,dense,239,0.726,0.952,0.968,0.968,0.965,0.976,0.984,0.894,0.710,3216.855,0
22,900/0,e5-large,dense,217,0.726,0.935,0.968,0.968,0.962,0.976,0.984,0.894,0.671,3007.177,0
40,1400/1,bge-m3,dense,171,0.694,0.952,0.968,1.000,0.952,0.968,0.968,0.842,0.645,4649.194,0


### 9.1 Effet de chaque facteur (moyennes marginales)

Sert à **justifier** chaque choix séparément : quelle taille de chunk, quel overlap, quel modèle, quelle méthode.
Les moyennes mélangent d'autres facteurs : elles indiquent une tendance, pas une preuve.

In [ ]:
for factor in ["chunking", "embedder", "method"]:
    print(f"\n--- complétude / MRR moyens par {factor} ---")
    display(exp.groupby(factor)[["rep@1", "rep@3", "rep@5", "rep@8", "cov@5", "hit@5", "MRR", "ctx@5"]].mean().round(3))
print("\n--- overlap (0 vs 1 élément) ---")
exp["overlap"] = exp.chunking.str.split("/").str[1]; exp["max_chars"] = exp.chunking.str.split("/").str[0]
display(exp[exp.method != "bm25"].groupby("overlap")[["rep@5", "cov@5", "MRR", "n_chunks"]].mean().round(3))


--- complétude / MRR moyens par chunking ---


,rep@1,rep@3,rep@5,rep@8,cov@5,hit@5,MRR,ctx@5
chunking,,,,,,,,
1400/0,0.677,0.906,0.954,0.975,0.956,0.959,0.813,4443.551
1400/1,0.696,0.910,0.963,0.984,0.967,0.970,0.827,4572.654
500/0,0.703,0.836,0.885,0.931,0.921,0.961,0.854,1893.680
500/1,0.710,0.834,0.885,0.919,0.918,0.952,0.852,1964.122
900/0,0.673,0.882,0.924,0.949,0.951,0.963,0.851,3193.320
900/1,0.677,0.876,0.931,0.961,0.954,0.965,0.849,3336.684



--- complétude / MRR moyens par embedder ---


,rep@1,rep@3,rep@5,rep@8,cov@5,hit@5,MRR,ctx@5
embedder,,,,,,,,
-,0.656,0.825,0.860,0.884,0.883,0.909,0.777,3195.629
bge-m3,0.685,0.887,0.935,0.960,0.958,0.973,0.851,3307.281
e5-base,0.681,0.866,0.927,0.968,0.949,0.965,0.839,3266.858
e5-large,0.718,0.894,0.940,0.966,0.957,0.973,0.865,3147.054



--- complétude / MRR moyens par method ---


,rep@1,rep@3,rep@5,rep@8,cov@5,hit@5,MRR,ctx@5
method,,,,,,,,
bm25,0.656,0.825,0.860,0.884,0.883,0.909,0.777,3195.629
dense,0.685,0.882,0.942,0.976,0.960,0.974,0.849,3211.229
hybrid,0.704,0.883,0.927,0.953,0.950,0.967,0.854,3269.565



--- overlap (0 vs 1 élément) ---


,rep@5,cov@5,MRR,n_chunks
overlap,,,,
0,0.931,0.952,0.850,251.000
1,0.937,0.957,0.853,268.667


## 10. Sélection de la configuration et de `k` (règle de la section 0, **dev uniquement**)



In [ ]:
n_dev = len(dev_in)
elig = exp[exp.method.isin(["dense", "hybrid"])].copy()
assert len(elig), "aucun embedder n'a pu être évalué"
best_rep = max(elig[f"rep@{k}"].max() for k in KS_SEL)
target = best_rep - 1.0 / n_dev - 1e-9
cands = []
for idx, row in elig.iterrows():
    for k in KS_SEL:
        if all(row[f"rep@{kk}"] >= target for kk in KS_SEL if kk >= k):      # complétude stable à partir de k
            cands.append({"idx": idx, "k": k, "chunking": row.chunking, "embedder": row.embedder, "method": row.method,
                          "rep": row[f"rep@{k}"], "cov": row[f"cov@{k}"], "ctx": row[f"ctx@{k}"], "MRR": row.MRR})
            break                                                              # k minimal pour cette configuration
cands = pd.DataFrame(cands).sort_values(["ctx", "cov", "MRR"], ascending=[True, False, False])
assert len(cands), "aucune configuration n'atteint la complétude cible"
print(f"Meilleur rep@k du dev = {best_rep:.3f} (n={n_dev}) ; cible ≥ {best_rep - 1.0 / n_dev:.3f} ; {len(cands)} configuration(s) admissible(s).")
display(cands.drop(columns="idx").round(3).head(10))
win = cands.iloc[0]
chosen = elig.loc[win["idx"]]
K_FINAL = int(win["k"])
print("\nCONFIGURATION RETENUE :", dict(chunking=chosen["chunking"], embedder=chosen["embedder"], method=chosen["method"]), "| k =", K_FINAL)

CH_CFG = prepared[chosen["chunking"]]["cfg"]; CH_ID = chosen["chunking"]; EMB_NAME = chosen["embedder"]; METHOD = chosen["method"]
mean_chunk = np.mean([c["n_chars"] for c in prepared[CH_ID]["chunks"]])
print("Courbe de la config retenue :", {k: dict(rep=round(chosen[f"rep@{k}"], 3), cov=round(chosen[f"cov@{k}"], 3)) for k in KS})
print(f"k retenu = {K_FINAL} -> ~{chosen[f'ctx@{K_FINAL}']:.0f} caractères de contexte en moyenne (~{chosen[f'ctx@{K_FINAL}'] / 3.3:.0f} tokens) : compatible avec le contexte de Qwen2.5-7B (32 k).")
lo, hi = wilson(round(chosen[f"rep@{K_FINAL}"] * n_dev), n_dev)
print(f"rep@{K_FINAL} = {chosen[f'rep@{K_FINAL}']:.0%}  (IC95 Wilson [{lo:.0%}-{hi:.0%}], n={n_dev}) | cov@{K_FINAL} = {chosen[f'cov@{K_FINAL}']:.0%}")
single = [i for i in range(n_dev) if len(dev_in.key_facts[i]) == 1]; multi = [i for i in range(n_dev) if len(dev_in.key_facts[i]) > 1]
print(f"(dont questions à un fait : {len(single)} ; multi-faits : {len(multi)})")

Meilleur rep@k du dev = 1.000 (n=62) ; cible ≥ 0.984 ; 15 configuration(s) admissible(s).


,k,chunking,embedder,method,rep,cov,ctx,MRR
11,5,900/1,bge-m3,dense,0.984,0.984,3419.097,0.852
3,5,1400/0,e5-base,dense,0.984,0.984,4413.048,0.787
9,5,1400/1,e5-large,hybrid,0.984,0.984,4505.452,0.824
4,5,1400/1,e5-base,dense,0.984,0.996,4517.065,0.795
14,5,1400/1,bge-m3,hybrid,0.984,0.984,4705.790,0.869
5,5,1400/1,e5-base,hybrid,0.984,0.984,4730.306,0.841
0,8,900/0,e5-base,dense,0.984,0.996,5187.419,0.817
10,8,900/0,bge-m3,dense,0.984,0.996,5271.452,0.838
1,8,900/1,e5-base,dense,0.984,0.984,5373.435,0.834
2,8,900/1,e5-base,hybrid,0.984,0.997,5401.177,0.869



CONFIGURATION RETENUE : {'chunking': '900/1', 'embedder': 'bge-m3', 'method': 'dense'} | k = 5
Courbe de la config retenue : {1: {'rep': np.float64(0.677), 'cov': np.float64(0.728)}, 3: {'rep': np.float64(0.887), 'cov': np.float64(0.909)}, 5: {'rep': np.float64(0.984), 'cov': np.float64(0.984)}, 8: {'rep': np.float64(1.0), 'cov': np.float64(1.0)}, 10: {'rep': np.float64(1.0), 'cov': np.float64(1.0)}}
k retenu = 5 -> ~3419 caractères de contexte en moyenne (~1036 tokens) : compatible avec le contexte de Qwen2.5-7B (32 k).
rep@5 = 98%  (IC95 Wilson [91%-100%], n=62) | cov@5 = 98%
(dont questions à un fait : 36 ; multi-faits : 26)


### 10.1 Échecs sur le dev (pour comprendre ce qui manque, pas pour régler davantage)

In [ ]:
pr = prepared[CH_ID]; P, Q_in, Q_ooc = store[(CH_ID, EMB_NAME)]
sc = Q_in @ P.T; dense_rank = topn(sc, 50)
ranked = dense_rank[:, :TOP_K_MAX] if METHOD == "dense" else np.array([rrf_fuse([d, b]) for d, b in zip(dense_rank, pr["bm_rank"])])
fails = [i for i in range(n_dev) if not pr["fm"][i][:, ranked[i, :K_FINAL]].any(axis=1).all()]
print(f"{len(fails)} question(s) du dev dont tous les faits-clés ne sont pas dans le top-{K_FINAL} :")
for i in fails:
    r = dev_in.iloc[i]
    missing = np.where(~pr["fm"][i][:, ranked[i, :K_FINAL]].any(axis=1))[0]
    where = [j for j in np.where(pr["fm"][i][missing].any(axis=0))[0]][:2]
    print(f"- [{r.id}] {r.question}\n    fiches de référence {r.gold_fiches} | fiches retrouvées {[pr['chunks'][j]['fiche'] for j in ranked[i, :K_FINAL]]}"
          f" | chunk contenant la preuve : {[pr['chunks'][j]['chunk_id'] + ' (' + cite(pr['chunks'][j]) + ')' for j in where]}")

1 question(s) du dev dont tous les faits-clés ne sont pas dans le top-5 :
- [d01] Quelles fonctions le guide recommande-t-il de nommer pour suivre au quotidien la sécurité et la protection des données ?
    fiches de référence [1] | fiches retrouvées [0, 0, 15, 1, 11] | chunk contenant la preuve : ['c0012 (Fiche 1 « Piloter la sécurité des données », Pour aller plus loin, p. 5)']


### 10.2 Aide à l'abstention (pour l'étape 5) : la similarité du meilleur chunk sépare-t-elle questions en corpus / hors corpus ?


In [ ]:
def auroc(pos, neg):
    pos, neg = np.asarray(pos), np.asarray(neg)
    return float(np.mean([(p > n) + 0.5 * (p == n) for p in pos for n in neg]))

top_in = (Q_in @ P.T).max(axis=1); top_ooc = (Q_ooc @ P.T).max(axis=1)
near_mask = (dev_ooc.category == "out_corpus_near").values
print("cosinus max | en corpus      :", np.round(np.percentile(top_in, [0, 25, 50, 75, 100]), 3))
print("cosinus max | hors-c. proche :", np.round(np.percentile(top_ooc[near_mask], [0, 25, 50, 75, 100]), 3))
print("cosinus max | hors-c. sujet  :", np.round(np.percentile(top_ooc[~near_mask], [0, 25, 50, 75, 100]), 3))
print(f"AUROC en corpus vs hors corpus : tous {auroc(top_in, top_ooc):.2f} | proches seulement {auroc(top_in, top_ooc[near_mask]):.2f}")

cands = np.unique(np.concatenate([top_in, top_ooc]))
def youden(t): return (top_in >= t).mean() + (top_ooc < t).mean() - 1
SIM_THRESHOLD = float(max(cands, key=youden))
print(f"Seuil (Youden sur le dev) : {SIM_THRESHOLD:.3f} -> garde {(top_in >= SIM_THRESHOLD).mean():.0%} des questions en corpus, "
      f"rejette {(top_ooc < SIM_THRESHOLD).mean():.0%} des hors corpus (dont {(top_ooc[near_mask] < SIM_THRESHOLD).mean():.0%} des proches)")

cosinus max | en corpus      : [0.546 0.618 0.67  0.706 0.767]
cosinus max | hors-c. proche : [0.348 0.519 0.569 0.601 0.652]
cosinus max | hors-c. sujet  : [0.309 0.314 0.32  0.334 0.348]
AUROC en corpus vs hors corpus : tous 0.92 | proches seulement 0.89
Seuil (Youden sur le dev) : 0.610 -> garde 81% des questions en corpus, rejette 92% des hors corpus (dont 90% des proches)


## 11. Construction et sauvegarde de l'index final (FAISS)

In [ ]:
final_chunks = prepared[CH_ID]["chunks"]
embedder = make_embedder(EMB_NAME)
t0 = time.time()
E = embedder.encode_passages([c["embed_text"] for c in final_chunks])
print(f"{len(final_chunks)} passages encodés en {time.time()-t0:.1f}s | dimension {E.shape[1]}")
index = faiss.IndexFlatIP(E.shape[1]); index.add(E)

faiss.write_index(index, str(OUTPUT_DIR / "faiss.index"))
np.save(OUTPUT_DIR / "embeddings.npy", E)
with open(OUTPUT_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
    for c in final_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
index_cfg = {
    "step": "2_index", "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "embedder": EMB_NAME, "embedder_hf_id": EMBEDDERS[EMB_NAME]["hf_id"] if EMB_NAME in EMBEDDERS else EMB_NAME,
    "method": METHOD, "chunking": CH_CFG, "k_final": int(K_FINAL), "sim_threshold": SIM_THRESHOLD, "rrf_k": 60,
    "n_chunks": len(final_chunks), "index_checklist": INDEX_CHECKLIST,
    "pdf_sha256": PDF_SHA256, "dev_sha256": DEV_SHA256, "test_sha256": TEST_SET_SHA256, "seed": SEED,
    "dev_hit@5": float(chosen["hit@5"]), "dev_MRR": float(chosen["MRR"]),
    "dev_rep@k_final": float(chosen[f"rep@{K_FINAL}"]), "dev_cov@k_final": float(chosen[f"cov@{K_FINAL}"]), "k_max": K_MAX, "selection_rule": "rep@k stable, ctx minimal (dev only)",
}
json.dump(index_cfg, open(OUTPUT_DIR / "index_config.json", "w"), indent=2, ensure_ascii=False)
print("Écrit dans", OUTPUT_DIR.resolve(), ":", sorted(p.name for p in OUTPUT_DIR.iterdir()))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

239 passages encodés en 2.0s | dimension 1024
Écrit dans /content/outputs/step2_index : ['chunks.jsonl', 'embeddings.npy', 'experiments_dev.csv', 'faiss.index', 'index_config.json']


## 12. `Retriever` réutilisable 

In [ ]:
class Retriever:
    def __init__(self, out_dir=OUTPUT_DIR, embedder=None):
        out_dir = Path(out_dir)
        self.cfg = json.load(open(out_dir / "index_config.json", encoding="utf-8"))
        self.chunks = load_jsonl(out_dir / "chunks.jsonl")
        self.index = faiss.read_index(str(out_dir / "faiss.index"))
        self.embedder = embedder or make_embedder(self.cfg["embedder"])
        self.bm25 = BM25([tokenize(c["embed_text"]) for c in self.chunks]) if self.cfg["method"] == "hybrid" else None
        self.k = self.cfg["k_final"]
    def search_batch(self, questions, k=None):
        k = k or self.k
        n = min(50, len(self.chunks))
        Q = self.embedder.encode_queries(list(questions))
        scores, ids = self.index.search(Q, n)
        out = []
        for q, s_row, i_row in zip(questions, scores, ids):
            if self.bm25 is not None:
                bm_rank = np.argsort(-self.bm25.scores(tokenize(q)))[:n]
                order = rrf_fuse([i_row, bm_rank], k=self.cfg["rrf_k"], topn=k)
            else:
                order = list(i_row[:k])
            cos = dict(zip(i_row.tolist(), s_row.tolist()))
            res = []
            for rank, j in enumerate(order, 1):
                c = self.chunks[j]
                cj = cos[int(j)] if int(j) in cos else float(Q[len(out)] @ self.index.reconstruct(int(j)))
                res.append({"rank": rank, "chunk_id": c["chunk_id"], "cosine": cj, "fiche": c["fiche"], "pages": c["pages"],
                            "citation": cite(c), "text": c["text"]})
            out.append({"question": q, "top1_cosine": float(s_row[0]), "results": res})
        return out
    def search(self, question, k=None):
        return self.search_batch([question], k)[0]

retriever = Retriever(OUTPUT_DIR, embedder=embedder)      # on réutilise l'embedder déjà chargé (évite un second chargement)
demo = retriever.search("Dans quel délai faut-il notifier une violation de données à la CNIL ?")
print(f"top-1 cosinus = {demo['top1_cosine']:.3f}")
for r in demo["results"]:
    print(f"#{r['rank']} [{r['chunk_id']}] cos={r['cosine']:.3f} | {r['citation']}\n    {textwrap.shorten(r['text'].replace(chr(10), ' '), 230)}")

# Contrôle de cohérence : FAISS == produit matriciel numpy
q_chk = embedder.encode_queries(queries_in[:5])
assert np.array_equal(index.search(q_chk, 5)[1], topn(q_chk @ E.T, 5)), "FAISS et numpy divergent"
print("OK : FAISS et numpy donnent les mêmes top-5.")

top-1 cosinus = 0.732
#1 [c0146] cos=0.732 | Fiche 19 « Gérer les incidents et les violations », Les précautions élémentaires, p. 42
    Tenir un registre interne de toutes les violations de données personnelles. Notifier à la CNIL, dans les 72 heures (tel que prévu par le RGPD), les violations présentant un risque pour les droits et libertés des personnes [...]
#2 [c0147] cos=0.718 | Fiche 19 « Gérer les incidents et les violations », Ce qu'il ne faut pas faire, p. 42
    Attendre que les personnes concernées ou des tiers détectent un incident et le signalent. Omettre l'analyse des risques qu'une violation de données personnelles pourrait avoir pour les droits et libertés des personnes. [...]
#3 [c0225] cos=0.684 | Fiche 19 « Gérer les incidents et les violations », Notes de bas de page, p. 42
    [80] La procédure de notification est détaillée sur le site de la CNIL (voir « Notifier une violation de données personnelles », cnil.fr). [81] Articles 33 et 34 du RGPD. [82] L'obligation d

## 13. Retrieval sur le jeu de test — **reporting uniquement**


In [ ]:
res_test = retriever.search_batch(test_in.question.tolist(), k=TOP_K_MAX)
def coverage(texts, key_facts):
    blob = norm(" ".join(texts))
    hits = [any(re.search(norm(a), blob) for a in alts) for alts in key_facts]
    return sum(hits) / len(hits), all(hits)

rows_t = []
for k in KS:
    fh, fp, cov, ans = [], [], [], []
    for (_, r), rr in zip(test_in.iterrows(), res_test):
        top = rr["results"][:k]
        fh.append(any(x["fiche"] in r.gold_fiches for x in top))
        fp.append(np.mean([x["fiche"] in r.gold_fiches for x in top]))
        c, a = coverage([x["text"] for x in top], r.key_facts)
        cov.append(c); ans.append(a)
    rows_t.append({"k": k, "fiche_hit@k (recall)": np.mean(fh), "fiche_precision@k": np.mean(fp), "couverture@k": np.mean(cov), "répondable@k": np.mean(ans)})
test_report = pd.DataFrame(rows_t).set_index("k")
display(test_report.round(3))
test_report.to_csv(OUTPUT_DIR / "retrieval_test_report.csv")

# Questions du test dont les faits ne sont pas tous dans le top-K_FINAL (cas d'échec attendus en B)
bad = []
for (_, r), rr in zip(test_in.iterrows(), res_test):
    c, a = coverage([x["text"] for x in rr["results"][:K_FINAL]], r.key_facts)
    if not a:
        bad.append((r.id, round(c, 2), r.gold_fiches, [x["fiche"] for x in rr["results"][:K_FINAL]], r.question))
print(f"\n{len(bad)}/{len(test_in)} questions du test dont tous les faits-clés ne sont pas dans le top-{K_FINAL} :")
for b in bad:
    print(f"- [{b[0]}] couverture={b[1]} | fiches de référence {b[2]} | fiches retrouvées {b[3]}\n    {b[4]}")

,fiche_hit@k (recall),fiche_precision@k,couverture@k,répondable@k
k,,,,
1,0.97,0.970,0.783,0.758
3,0.97,0.818,0.889,0.879
5,1.00,0.703,1.000,1.000
8,1.00,0.564,1.000,1.000
10,1.00,0.527,1.000,1.000



0/33 questions du test dont tous les faits-clés ne sont pas dans le top-5 :


### 13.1 Questions hors corpus du test : que ferait le seuil de similarité (calibré sur le dev) ?

In [ ]:
res_ooc = retriever.search_batch(test_ooc.question.tolist(), k=1)
t_ooc = np.array([r["top1_cosine"] for r in res_ooc]); t_in = np.array([r["top1_cosine"] for r in res_test])
near_t = (test_ooc.category == "out_corpus_near").values
print(f"Seuil du dev = {SIM_THRESHOLD:.3f}")
print(f"En corpus conservées     : {(t_in >= SIM_THRESHOLD).mean():.0%}")
print(f"Hors corpus rejetées     : {(t_ooc < SIM_THRESHOLD).mean():.0%} (proches : {(t_ooc[near_t] < SIM_THRESHOLD).mean():.0%}, hors sujet : {(t_ooc[~near_t] < SIM_THRESHOLD).mean():.0%})")
print(f"AUROC test (en corpus vs hors corpus) : {auroc(t_in, t_ooc):.2f} | proches seulement : {auroc(t_in, t_ooc[near_t]):.2f}")
pd.DataFrame({"id": test_ooc.id, "category": test_ooc.category, "top1_cosine": t_ooc, "rejetée": t_ooc < SIM_THRESHOLD}).to_csv(OUTPUT_DIR / "retrieval_test_ooc_scores.csv", index=False)

Seuil du dev = 0.610
En corpus conservées     : 76%
Hors corpus rejetées     : 65% (proches : 54%, hors sujet : 100%)
AUROC test (en corpus vs hors corpus) : 0.84 | proches seulement : 0.79
